In [1]:
import pandas as pd

df = pd.read_csv(
    "/mnt/c/Users/Dhruv/OneDrive/Desktop/samsung/aqi/aqi.csv"
)

print(df)

                 datetime        date  year  month  day  hour  day_of_week  \
0     2020-01-01 06:00:00  2020-01-01  2020      1    1     6            2   
1     2020-01-01 12:00:00  2020-01-01  2020      1    1    12            2   
2     2020-01-01 18:00:00  2020-01-01  2020      1    1    18            2   
3     2020-01-01 23:00:00  2020-01-01  2020      1    1    23            2   
4     2020-01-02 06:00:00  2020-01-02  2020      1    2     6            3   
...                   ...         ...   ...    ...  ...   ...          ...   
8763  2025-12-30 23:00:00  2025-12-30  2025     12   30    23            1   
8764  2025-12-31 06:00:00  2025-12-31  2025     12   31     6            2   
8765  2025-12-31 12:00:00  2025-12-31  2025     12   31    12            2   
8766  2025-12-31 18:00:00  2025-12-31  2025     12   31    18            2   
8767  2025-12-31 23:00:00  2025-12-31  2025     12   31    23            2   

      is_weekend   pm25   pm10  ...    co    o3  temperature  h

In [2]:
df

,datetime,date,year,month,day,hour,day_of_week,is_weekend,pm25,pm10,...,co,o3,temperature,humidity,wind_speed,visibility,aqi,season_post_monsoon,season_summer,season_winter
0,2020-01-01 06:00:00,2020-01-01,2020,1,1,6,2,0,327.0,596.1,...,5.13,15.2,9.9,100,2.2,1.3,500,0,0,1
1,2020-01-01 12:00:00,2020-01-01,2020,1,1,12,2,0,231.1,436.6,...,3.92,25.1,17.2,46,9.9,2.6,408,0,0,1
2,2020-01-01 18:00:00,2020-01-01,2020,1,1,18,2,0,271.6,483.2,...,5.75,25.6,14.1,64,6.6,2.5,466,0,0,1
3,2020-01-01 23:00:00,2020-01-01,2020,1,1,23,2,0,324.8,655.0,...,4.49,17.5,12.5,45,4.4,0.7,500,0,0,1
4,2020-01-02 06:00:00,2020-01-02,2020,1,2,6,3,0,350.2,636.3,...,4.44,15.3,6.3,100,2.4,0.8,500,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8763,2025-12-30 23:00:00,2025-12-30,2025,12,30,23,1,0,272.5,574.7,...,4.77,22.4,14.3,49,6.8,2.9,500,0,0,1
8764,2025-12-31 06:00:00,2025-12-31,2025,12,31,6,2,0,308.5,666.1,...,5.50,16.8,8.2,100,2.5,0.8,500,0,0,1
8765,2025-12-31 12:00:00,2025-12-31,2025,12,31,12,2,0,183.5,369.0,...,2.99,28.5,21.3,63,8.8,3.1,349,0,0,1
8766,2025-12-31 18:00:00,2025-12-31,2025,12,31,18,2,0,260.0,450.7,...,5.53,25.1,16.4,61,5.9,2.9,426,0,0,1


In [5]:
import pandas as pd

# Convert and sort chronologically
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)



# AQI lags used in your previous paper
# 1 = 6h, 2 = 12h, 4 = 24h, 8 = 48h, 28 = 1 week
for lag in [1, 2, 4, 8, 28]:
    df[f"aqi_lag_{lag}"] = df["aqi"].shift(lag)

# Final dataset — NO meteorological variables
final_cols = [
    # Temporal
    "year",
    "month",
    "day",
    "hour",
    "day_of_week",
    "is_weekend",

    # Encoded season
    "season_post_monsoon",
    "season_summer",
    "season_winter",

    # Pollutants
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3",

    # AQI historical/persistence
    "aqi_lag_1",
    "aqi_lag_2",
    "aqi_lag_4",
    "aqi_lag_8",
    "aqi_lag_28",

    # Target
    "aqi"
]

df_model = df[final_cols].copy()

# Remove rows where lag features are unavailable
df_model = df_model.dropna().reset_index(drop=True)

# Save untouched master dataset
df_model.to_csv(
    "/mnt/c/Users/Dhruv/OneDrive/Desktop/samsung/aqi/AQI_master_clean.csv",
    index=False
)

print("Shape:", df_model.shape)
print("\nColumns:")
print(df_model.columns.tolist())

print("\nMissing values:")
print(df_model.isnull().sum())

Shape: (8740, 21)

Columns:
['year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'season_post_monsoon', 'season_summer', 'season_winter', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_4', 'aqi_lag_8', 'aqi_lag_28', 'aqi']

Missing values:
year                   0
month                  0
day                    0
hour                   0
day_of_week            0
is_weekend             0
season_post_monsoon    0
season_summer          0
season_winter          0
pm25                   0
pm10                   0
no2                    0
so2                    0
co                     0
o3                     0
aqi_lag_1              0
aqi_lag_2              0
aqi_lag_4              0
aqi_lag_8              0
aqi_lag_28             0
aqi                    0
dtype: int64


In [6]:
df["datetime"] = pd.to_datetime(df["datetime"])

print(
    df["datetime"]
    .sort_values()
    .diff()
    .value_counts()
    .head(10)
)

datetime
0 days 06:00:00    4384
0 days 05:00:00    2192
0 days 07:00:00    2191
Name: count, dtype: int64


In [7]:
# Check duplicate timestamps
print("Duplicate datetime rows:", df["datetime"].duplicated().sum())

# Show examples of duplicated timestamps
print(
    df[df["datetime"].duplicated(keep=False)]
    .sort_values("datetime")[["datetime", "aqi"]]
    .head(20)
)

Duplicate datetime rows: 0
Empty DataFrame
Columns: [datetime, aqi]
Index: []


In [8]:
print(df.columns.tolist())

['datetime', 'date', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed', 'visibility', 'aqi', 'season_post_monsoon', 'season_summer', 'season_winter', 'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_4', 'aqi_lag_8', 'aqi_lag_28']


In [9]:
print(df.shape)
print(df.head())

(8768, 27)
             datetime        date  year  month  day  hour  day_of_week  \
0 2020-01-01 06:00:00  2020-01-01  2020      1    1     6            2   
1 2020-01-01 12:00:00  2020-01-01  2020      1    1    12            2   
2 2020-01-01 18:00:00  2020-01-01  2020      1    1    18            2   
3 2020-01-01 23:00:00  2020-01-01  2020      1    1    23            2   
4 2020-01-02 06:00:00  2020-01-02  2020      1    2     6            3   

   is_weekend   pm25   pm10  ...  visibility  aqi  season_post_monsoon  \
0           0  327.0  596.1  ...         1.3  500                    0   
1           0  231.1  436.6  ...         2.6  408                    0   
2           0  271.6  483.2  ...         2.5  466                    0   
3           0  324.8  655.0  ...         0.7  500                    0   
4           0  350.2  636.3  ...         0.8  500                    0   

   season_summer  season_winter  aqi_lag_1  aqi_lag_2  aqi_lag_4  aqi_lag_8  \
0              0    

In [10]:
import pandas as pd
import numpy as np
import os

# ============================================================
# LOAD MASTER CLEAN DATASET
# ============================================================

master_path = "/mnt/c/Users/Dhruv/OneDrive/Desktop/samsung/aqi/aqi.csv"

df = pd.read_csv(master_path)

# Pollutants where missingness will be introduced
pollutants = [
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3"
]

# Missingness levels
missing_levels = [5, 10, 20, 30]

# Fixed seed for reproducibility
RANDOM_SEED = 42

# Output directory
output_dir = "/mnt/c/Users/Dhruv/OneDrive/Desktop/samsung/aqi"

# ============================================================
# CREATE RANDOM MISSINGNESS
# ============================================================

for percentage in missing_levels:

    df_missing = df.copy()

    for column in pollutants:

        # Number of values to make missing
        n_missing = int(len(df_missing) * percentage / 100)

        # Randomly select row indices
        rng = np.random.default_rng(
            RANDOM_SEED + percentage + pollutants.index(column)
        )

        missing_indices = rng.choice(
            df_missing.index,
            size=n_missing,
            replace=False
        )

        # Introduce NaN
        df_missing.loc[missing_indices, column] = np.nan

    # Save dataset
    output_path = os.path.join(
        output_dir,
        f"AQI_missing_{percentage}pct.csv"
    )

    df_missing.to_csv(output_path, index=False)

    print(f"Created: AQI_missing_{percentage}pct.csv")

Created: AQI_missing_5pct.csv
Created: AQI_missing_10pct.csv
Created: AQI_missing_20pct.csv
Created: AQI_missing_30pct.csv


In [11]:
for percentage in [5, 10, 20, 30]:

    path = f"/mnt/c/Users/Dhruv/OneDrive/Desktop/samsung/aqi/AQI_missing_{percentage}pct.csv"

    test_df = pd.read_csv(path)

    print(f"\n===== {percentage}% MISSINGNESS =====")

    print(test_df[pollutants].isnull().sum())

    print(
        "Total missing pollutant values:",
        test_df[pollutants].isnull().sum().sum()
    )


===== 5% MISSINGNESS =====
pm25    438
pm10    438
no2     438
so2     438
co      438
o3      438
dtype: int64
Total missing pollutant values: 2628

===== 10% MISSINGNESS =====
pm25    876
pm10    876
no2     876
so2     876
co      876
o3      876
dtype: int64
Total missing pollutant values: 5256

===== 20% MISSINGNESS =====
pm25    1753
pm10    1753
no2     1753
so2     1753
co      1753
o3      1753
dtype: int64
Total missing pollutant values: 10518

===== 30% MISSINGNESS =====
pm25    2630
pm10    2630
no2     2630
so2     2630
co      2630
o3      2630
dtype: int64
Total missing pollutant values: 15780
